# Introduction

In [1]:
# SYSTEM
import os
import sys
from pathlib import Path

# DATA WRANGLING, STATISTICS AND VISUALIZATION
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform, loguniform

# MODEL SELECTION
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from optuna import visualization as vis

# MODEL EVALUATION
from sklearn.metrics import classification_report, confusion_matrix

# HYPERPARAMETER TUNING
import optuna
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution

Assess `accuracy`, `precision`, `recall`, `f1-score`:

In [2]:
# Evaluation metrics considered
metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'AUC': 'roc_auc'
}

# Number of models trained per search
j = 1000

# Data

In [3]:
# Are we working on google colab?
if 'google.colab' in str(get_ipython()):
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    # If so, look for the files on the drive location
    data_path = Path('/content/drive/MyDrive/TFM/data')
    
else:
    
    # If the session is running local
    root_path = Path(os.path.abspath(".."))
    
    # Add path to Python's system
    if str(root_path) not in sys.path:
        sys.path.append(str(root_path))
    
    # Create file path
    file_path = root_path / 'data' / 'intermediate' / 'clinical_data_explored.parquet'

Load the data, check the structure and split the data into target class and features:

In [4]:
# Load data
df = pd.read_parquet(file_path)

# Check general structure
df.info()

# Features
X = df.drop("af_recurrence", axis=1)

# Target class manually encoded
y = df["af_recurrence"].map({"no":0, "yes":1})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   gender              720 non-null    category
 1   age                 720 non-null    float64 
 2   smoking_status      720 non-null    category
 3   working_status      720 non-null    category
 4   education_level     720 non-null    category
 5   household_size      720 non-null    float64 
 6   children            720 non-null    float64 
 7   marital_status      720 non-null    category
 8   cohort              720 non-null    category
 9   node                720 non-null    category
 10  bmi                 720 non-null    float64 
 11  total_met           720 non-null    float64 
 12  t2_diabetes         720 non-null    category
 13  hypolipidemics      703 non-null    category
 14  hypercholesterol    720 non-null    category
 15  hbp                 720 non-null    cate

## Preprocessing

Divide data set into train and test set:

In [5]:
# Divide into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
    )

Create the preprocessor object:

In [6]:
from src.models.data_preprocessing import preprocess

# Create the preprocessing pipeline
preprocessor = preprocess(X)

# Display it
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

# Train models and optimize hyperparameters

## Random forest

Ensemble the model:

In [7]:
# Full pipeline
pipe_RF = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

# Hyperparameters search space
params_dist_RF = {
    'clf__n_estimators': IntDistribution(50, 300),
    'clf__max_depth': IntDistribution(2, 32),
    'clf__min_samples_split': IntDistribution(2, 20), 
    'clf__criterion': CategoricalDistribution(['gini', 'entropy']),
    'clf__class_weight': CategoricalDistribution([None, 'balanced', 'balanced_subsample'])
}

# Random search
optuna_search_RF = OptunaSearchCV(
    estimator=pipe_RF,
    param_distributions=params_dist_RF,
    n_trials=50,
    cv=5,
    scoring="recall",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

C:\Users\Miguel\AppData\Local\Temp\ipykernel_856\1253929377.py:17: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  optuna_search_RF = OptunaSearchCV(


Train and optimize the model:

In [8]:
optuna_search_RF.fit(X_train, y_train)

[I 2026-04-17 23:38:21,618] A new study created in memory with name: no-name-afd64f24-2019-472b-b958-e2ea29a8d760
[I 2026-04-17 23:38:30,981] Trial 6 finished with value: 0.2392566782810685 and parameters: {'clf__n_estimators': 58, 'clf__max_depth': 17, 'clf__min_samples_split': 6, 'clf__criterion': 'entropy', 'clf__class_weight': 'balanced'}. Best is trial 6 with value: 0.2392566782810685.
[I 2026-04-17 23:38:35,044] Trial 5 finished with value: 0.3207897793263647 and parameters: {'clf__n_estimators': 100, 'clf__max_depth': 16, 'clf__min_samples_split': 14, 'clf__criterion': 'entropy', 'clf__class_weight': 'balanced'}. Best is trial 5 with value: 0.3207897793263647.
[I 2026-04-17 23:38:35,229] Trial 7 finished with value: 0.28246225319396057 and parameters: {'clf__n_estimators': 100, 'clf__max_depth': 26, 'clf__min_samples_split': 11, 'clf__criterion': 'entropy', 'clf__class_weight': 'balanced'}. Best is trial 5 with value: 0.3207897793263647.
[I 2026-04-17 23:38:40,962] Trial 1 finis

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'clf__class_weight': CategoricalDi...d_subsample')), 'clf__criterion': CategoricalDi...', 'entropy')), 'clf__max_depth': IntDistributi...low=2, step=1), 'clf__min_samples_split': IntDistributi...low=2, step=1), ...}"
,cv,5
,enable_pruning,False
,error_score,nan
,max_iter,1000
,n_jobs,-1
,n_trials,100
,random_state,42
,refit,True
,return_train_score,False


Check the optimization process along time:

In [9]:
# Get the study
study_RF = optuna_search_RF.study_

# Display the plot
fig = vis.plot_optimization_history(study_RF)
fig.show()

Evaluate the model:

In [10]:
# Extract the optimized model
optimized_RF = optuna_search_RF.best_estimator_

# Make predictions
y_pred_RF = optimized_RF.predict(X_test)

# Generate classification report
report_RF = classification_report(y_test, y_pred_RF)

# Display results
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_RF))
print("\nClassification Report:\n", report_RF)


Confusion Matrix:
 [[63 29]
 [25 27]]

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.68      0.70        92
           1       0.48      0.52      0.50        52

    accuracy                           0.62       144
   macro avg       0.60      0.60      0.60       144
weighted avg       0.63      0.63      0.63       144



## Support Vector Machine


Ensemble the model:

In [11]:
# Full pipeline
pipe_SVM = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', SVC(random_state=42))
])

# Parameters grid
params_dist_SVM = {
    'clf__C': FloatDistribution(1e-3, 1e2, log=True),
    'clf__kernel': CategoricalDistribution(['linear', 'rbf', 'poly', 'sigmoid']),
    'clf__gamma': CategoricalDistribution(['scale', 'auto']), 
    'clf__degree': IntDistribution(2, 5),
    'clf__class_weight': CategoricalDistribution([None, 'balanced', 'balanced_subsample'])
}

# Random search
optuna_search_SVM = OptunaSearchCV(
    estimator=pipe_SVM,
    param_distributions=params_dist_SVM,
    n_trials=50,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

C:\Users\Miguel\AppData\Local\Temp\ipykernel_856\1966857904.py:17: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  optuna_search_SVM = OptunaSearchCV(


Train and optimize the model:

In [12]:
optuna_search_SVM.fit(X_train, y_train)

[I 2026-04-17 23:43:27,298] A new study created in memory with name: no-name-bf3097a6-b9d4-4279-8058-6640271e8cbb
c:\Users\Miguel\miniconda3\envs\tfm\Lib\site-packages\optuna_integration\sklearn\sklearn.py:404: RuntimeWarning: Mean of empty slice
  trial.set_user_attr("mean_{}".format(name), np.nanmean(array))
c:\Users\Miguel\miniconda3\envs\tfm\Lib\site-packages\optuna_integration\sklearn\sklearn.py:404: RuntimeWarning: Mean of empty slice
  trial.set_user_attr("mean_{}".format(name), np.nanmean(array))
c:\Users\Miguel\miniconda3\envs\tfm\Lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Miguel\miniconda3\envs\tfm\Lib\site-packages\optuna_integration\sklearn\sklearn.py:404: RuntimeWarning: Mean of empty slice
  trial.set_user_attr("mean_{}".format(name), np.nanmean(array))
c:\Users\Miguel\miniconda3\envs\tfm\Lib\site-packages\numpy\lib\nanfunctions.py:1879: Runtime

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'clf__C': FloatDistribu...01, step=None), 'clf__class_weight': CategoricalDi...d_subsample')), 'clf__degree': IntDistributi...low=2, step=1), 'clf__gamma': CategoricalDi...ale', 'auto')), ...}"
,cv,5
,enable_pruning,False
,error_score,nan
,max_iter,1000
,n_jobs,-1
,n_trials,50
,random_state,42
,refit,True
,return_train_score,False


Time:

In [13]:
# Get the study
study_SVM = optuna_search_SVM.study_

# Display the plot
fig = vis.plot_optimization_history(study_SVM)
fig.show()

Evaluate the model:

In [14]:
# Extract the optimized model
optimized_SVM = optuna_search_SVM.best_estimator_

# Make predictions
y_pred_SVM = optimized_RF.predict(X_test)

# Generate classification report
report_SVM = classification_report(y_test, y_pred_SVM)

# Display results
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_SVM))
print("\nClassification Report:\n", report_SVM)

Confusion Matrix:
 [[63 29]
 [25 27]]

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.68      0.70        92
           1       0.48      0.52      0.50        52

    accuracy                           0.62       144
   macro avg       0.60      0.60      0.60       144
weighted avg       0.63      0.63      0.63       144



## K-Nearest Neighbors

In [15]:
# Full pipeline
pipe_KNN = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', KNeighborsClassifier())
])

# Parameters grid
params_dist_KNN = {
    'clf__n_neighbors': IntDistribution(3, 50),
    'clf__weights': CategoricalDistribution(['uniform', 'distance']),
    'clf__p': IntDistribution(1, 2),
    'clf__algorithm': CategoricalDistribution(['auto', 'ball_tree', 'kd_tree', 'brute'])
}

# Random search
optuna_search_KNN = OptunaSearchCV(
    estimator=pipe_KNN,
    param_distributions=params_dist_KNN,
    n_trials=50,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

C:\Users\Miguel\AppData\Local\Temp\ipykernel_856\2511157716.py:16: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  optuna_search_KNN = OptunaSearchCV(


Train and optimize the model:

In [16]:
optuna_search_KNN.fit(X_train, y_train)

[I 2026-04-17 23:43:40,825] A new study created in memory with name: no-name-3cc8e4c8-a94d-415c-93b3-f03236fcb7ec
[I 2026-04-17 23:43:41,902] Trial 3 finished with value: 0.28652729384436704 and parameters: {'clf__n_neighbors': 4, 'clf__weights': 'distance', 'clf__p': 2, 'clf__algorithm': 'kd_tree'}. Best is trial 3 with value: 0.28652729384436704.
[I 2026-04-17 23:43:42,064] Trial 2 finished with value: 0.1859465737514518 and parameters: {'clf__n_neighbors': 11, 'clf__weights': 'uniform', 'clf__p': 1, 'clf__algorithm': 'ball_tree'}. Best is trial 3 with value: 0.28652729384436704.
[I 2026-04-17 23:43:42,114] Trial 0 finished with value: 0.033449477351916376 and parameters: {'clf__n_neighbors': 36, 'clf__weights': 'uniform', 'clf__p': 1, 'clf__algorithm': 'kd_tree'}. Best is trial 3 with value: 0.28652729384436704.
[I 2026-04-17 23:43:43,087] Trial 8 finished with value: 0.06202090592334495 and parameters: {'clf__n_neighbors': 31, 'clf__weights': 'distance', 'clf__p': 2, 'clf__algorith

,estimator,Pipeline(step...lassifier())])
,param_distributions,"{'clf__algorithm': CategoricalDi...ee', 'brute')), 'clf__n_neighbors': IntDistributi...low=3, step=1), 'clf__p': IntDistributi...low=1, step=1), 'clf__weights': CategoricalDi..., 'distance'))}"
,cv,5
,enable_pruning,False
,error_score,nan
,max_iter,1000
,n_jobs,-1
,n_trials,50
,random_state,42
,refit,True
,return_train_score,False


In [17]:
# Get the study
study_KNN = optuna_search_KNN.study_

# Display the plot
fig = vis.plot_optimization_history(study_KNN)
fig.show()

Evaluate the model:

In [18]:
# Extract the optimized model
optimized_KNN = optuna_search_KNN.best_estimator_

# Make predictions
y_pred_KNN = optimized_KNN.predict(X_test)

# Generate classification report
report_KNN = classification_report(y_test, y_pred_KNN)

# Display results
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_KNN))
print("\nClassification Report:\n", report_KNN)

Confusion Matrix:
 [[65 27]
 [35 17]]

Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.71      0.68        92
           1       0.39      0.33      0.35        52

    accuracy                           0.57       144
   macro avg       0.52      0.52      0.52       144
weighted avg       0.55      0.57      0.56       144

